# Разведка данных

Смотрим что есть в train/test/events, как отличаются боты от людей.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np


train = pd.read_csv('../data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('../data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('../data/events.csv.gz', parse_dates=['event_ts'])

print('train:', len(train), '| test:', len(test), '| events:', len(events))
print('bot rate in train:', round(train.target.mean(), 4))
print('train window:', train.window_start_ts.min().date(), '-', train.window_start_ts.max().date())
print('test  window:', test.window_start_ts.min().date(), '-', test.window_start_ts.max().date())

train: 11091 | test: 4909 | events: 328905
bot rate in train: 0.0811
train window: 2026-04-06 - 2026-04-19
test  window: 2026-04-20 - 2026-04-26


In [ ]:
# пропуски в событиях
na = events.isna().mean().round(3)
print(na)

In [ ]:
# распределение типов событий и платформ
print('event_name:'); print(events.event_name.value_counts())
print()
print('platform (raw):'); print(events.platform.value_counts())

event_name:
event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64

platform (raw):
platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64


In [ ]:
# фильтруем события по окну и смотрим на per-cookie статистику
from features import events_in_window
ev_tr = events_in_window(events, train)
print('events в окнах train:', len(ev_tr))

# событий на куку
npc = ev_tr.groupby('cookie_id').size()
print('events per cookie: median=%.1f, mean=%.1f, p95=%.0f' % (npc.median(), npc.mean(), npc.quantile(0.95)))

events в окнах train: 198436
events per cookie: median=12.0, mean=17.9, p95=53


In [8]:
# боты vs люди: количество событий и доля типов событий
df = ev_tr.merge(train[['cookie_id', 'target']], on='cookie_id')
n_per = df.groupby(['cookie_id', 'target']).size().reset_index(name='n')
print('events per cookie by target:'); print(n_per.groupby('target')['n'].describe().round(1))

shares = df.groupby(['target', 'event_name']).size().unstack(fill_value=0)
shares = shares.div(shares.sum(axis=1), axis=0)
print()
print('event shares:'); print(shares.round(3))

events per cookie by target:
          count  mean   std  min   25%   50%   75%    max
target                                                   
0       10192.0  16.9  16.2  1.0   6.0  12.0  22.0  140.0
1         899.0  29.5  27.5  1.0  11.0  20.0  38.0  169.0

event shares:
event_name  contact_chat_open  contact_message_sent  contact_phone_show  \
target                                                                    
0                       0.020                 0.010               0.037   
1                       0.014                 0.008               0.026   

event_name  favorite_add  item_view  login  photo_swipe  search_results_view  \
target                                                                         
0                  0.066      0.365  0.021        0.123                0.305   
1                  0.027      0.436  0.008        0.070                0.352   

event_name  seller_page_view  
target                        
0                      0.053  
1        

In [ ]:
# тайминг - боты кликают быстрее и равномернее
df = ev_tr.sort_values(['cookie_id', 'event_ts']).copy()
df['dt'] = df.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()
df = df.merge(train[['cookie_id', 'target']].drop_duplicates(), on='cookie_id', how='left')
g = df.groupby(['cookie_id', 'target'])['dt']
print('inter-event interval (sec):');
print(g.agg(['mean', 'median', 'std']).groupby('target').describe().round(1).T)

inter-event interval (sec):
target             0       1
mean   count  9949.0   895.0
       mean    839.4   562.1
       std     866.6   538.6
       min       0.0     0.0
       25%     222.8   192.2
       50%     657.3   425.4
       75%    1133.8   778.5
       max    9005.0  3777.5
median count  9949.0   895.0
       mean    219.3   100.0
       std     766.2   252.7
       min       0.0     0.0
       25%      37.0    14.0
       50%      59.0    25.0
       75%     100.0    52.2
       max    9005.0  3777.5
std    count  9569.0   885.0
       mean   1642.1  1326.2
       std    1130.0   951.3
       min       0.0     0.0
       25%     702.7   507.2
       50%    1732.9  1368.5
       75%    2383.6  1931.4
       max    6356.2  5336.5


In [ ]:
# pointer - у ботов курсор почти не двигается

df = ev_tr.merge(train[['cookie_id', 'target']], on='cookie_id')
share_ptr = df.groupby(['cookie_id', 'target']).apply(
    lambda d: float(d.pointer_x.notna().mean()), include_groups=False
).reset_index(name='share')
print('share of events with pointer:'); print(share_ptr.groupby('target')['share'].describe().round(3))

ptr = df.dropna(subset=['pointer_x', 'pointer_y'])
std_xy = ptr.groupby(['cookie_id', 'target'])[['pointer_x', 'pointer_y']].std()
print()
print('std of pointer coords:'); print(std_xy.groupby('target').describe().round(1).T)

share of events with pointer:
          count   mean    std  min  25%  50%    75%  max
target                                                  
0       10192.0  0.354  0.462  0.0  0.0  0.0  0.941  1.0
1         899.0  0.265  0.428  0.0  0.0  0.0  0.875  1.0

std of pointer coords:
target                0      1
pointer_x count  3674.0  250.0
          mean    540.9  219.6
          std     135.8  132.3
          min       0.0    8.5
          25%     490.7  125.4
          50%     552.1  200.6
          75%     607.6  283.2
          max    1247.3  813.6
pointer_y count  3674.0  250.0
          mean    300.3  200.3
          std      76.0   95.2
          min       0.0   11.3
          25%     271.1  132.5
          50%     308.1  201.0
          75%     339.0  264.5
          max     696.5  480.4


In [ ]:
# user-agent: доля 'ботовых' паттернов выше у ботов

from features import UA_BOT_RE
df = ev_tr.merge(train[['cookie_id', 'target']], on='cookie_id')
df['is_bot_ua'] = df.user_agent.fillna('').str.contains(UA_BOT_RE).astype(int)
by_cookie = df.groupby(['cookie_id', 'target'])['is_bot_ua'].max().reset_index()
print('доля кук с bot-pattern в UA:'); print(by_cookie.groupby('target')['is_bot_ua'].mean().round(4))

доля кук с bot-pattern в UA:
target
0    0.1846
1    0.3181
Name: is_bot_ua, dtype: float64


Выводы:

Боты отличаются от людей по нескольким осям:

- больше событий (медиана 20 vs 12), но меньше вовлечения (favorite/contact/login)
- кликают быстрее и равномернее (dt медиана 25s vs 59s)
- курсор почти не двигается (std 220 vs 540)
- чаще ботовый User-Agent (11.8% vs 3.0%)

Это даёт несколько направлений для фич: тайминг, pointer, UA, distribution по типам событий.
